In [ ]:

!pip -q install torchmetrics

import os, copy, time, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torch.cuda.amp import autocast, GradScaler
from torchmetrics.classification import BinaryAUROC


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 21.0 MB/s eta 0:00:00


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# Your dataset root (edit if needed)
# Expected structure:
# /content/data/train/angiogram/*.jpg
# /content/data/train/non_angiogram/*.jpg
# /content/data/val/angiogram/*.jpg
# /content/data/val/non_angiogram/*.jpg
# /content/data/test/angiogram/*.jpg
# /content/data/test/non_angiogram/*.jpg
DATA_ROOT = "/content/drive/MyDrive/data"

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")


DEVICE: cpu


In [ ]:
BATCH_SIZE = 32
IMG_SIZE = 224
NUM_WORKERS = 2

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.RandomAffine(degrees=0, translate=(0.06, 0.06), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),  # helps across hospitals/contrast differences
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


In [ ]:
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds   = datasets.ImageFolder(VAL_DIR, transform=val_tfms)


print("Classes:", train_ds.classes)  # should be ['angiogram', 'non_angiogram'] OR similar
class_to_idx = train_ds.class_to_idx
print("class_to_idx:", class_to_idx)

# Count samples per class
targets = np.array([y for _, y in train_ds.samples])
class_counts = np.bincount(targets)
print("Train class counts:", class_counts)

# Weighted sampling helps if future you adds imbalance, and generally stabilizes training
class_weights = 1.0 / (class_counts + 1e-9)
sample_weights = class_weights[targets]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)



Classes: ['angiogram', 'non_angiogram']
class_to_idx: {'angiogram': 0, 'non_angiogram': 1}
Train class counts: [2539 2499]


In [ ]:

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds   = datasets.ImageFolder(VAL_DIR, transform=val_tfms)



In [ ]:
print("Classes:", train_ds.classes)  # should be ['angiogram', 'non_angiogram'] OR similar
class_to_idx = train_ds.class_to_idx
print("class_to_idx:", class_to_idx)


Classes: ['angiogram', 'non_angiogram']
class_to_idx: {'angiogram': 0, 'non_angiogram': 1}


In [ ]:
# Count samples per class
targets = np.array([y for _, y in train_ds.samples])
class_counts = np.bincount(targets)
print("Train class counts:", class_counts)

# Weighted sampling helps if future you adds imbalance, and generally stabilizes training
class_weights = 1.0 / (class_counts + 1e-9)
sample_weights = class_weights[targets]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

Train class counts: [2539 2499]


In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze backbone first for warmup
for p in model.parameters():
    p.requires_grad = False

# Replace classifier head (binary output as 1 logit)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 1)  # 1 logit
)

model = model.to(DEVICE)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 166MB/s]


In [ ]:
# pos_weight biases the model against false positives (helps precision for "angiogram")
# If angiogram is class 0/1 depends on folder order. We'll compute correctly.

angiogram_idx = class_to_idx.get("angiogram", None)
non_idx = class_to_idx.get("non_angiogram", None)

if angiogram_idx is None or non_idx is None:
    print("Warning: folder names not exactly 'angiogram'/'non_angiogram'. Adjust idx mapping accordingly.")

# We will treat "angiogram" as POSITIVE (label=1) for precision targeting.
# But ImageFolder labels are fixed by alphabetical folder order.
# So we map: y_pos = 1 if original label == angiogram_idx else 0
def map_to_pos(y):
    return 1 if y == angiogram_idx else 0

# Compute pos_weight for BCEWithLogitsLoss: pos_weight = (neg/pos)
pos_count = np.sum([map_to_pos(y) for _, y in train_ds.samples])
neg_count = len(train_ds) - pos_count
pos_weight = torch.tensor([neg_count / (pos_count + 1e-9)], dtype=torch.float32).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Only train the head first
optimizer = optim.AdamW(model.fc.parameters(), lr=3e-4, weight_decay=1e-4)

# Scheduler that reduces LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.4, patience=2)
scaler = GradScaler()



/tmp/ipython-input-3241988442.py:28: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


In [ ]:
def sigmoid(x): return 1 / (1 + np.exp(-x))

def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_probs, all_true = [], []
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            # map labels so angiogram=1
            yb = torch.tensor([map_to_pos(int(y)) for y in yb], dtype=torch.float32).to(DEVICE).view(-1, 1)

            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)

            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
            true  = yb.detach().cpu().numpy().reshape(-1)
            all_probs.append(probs)
            all_true.append(true)

    all_probs = np.concatenate(all_probs)
    all_true  = np.concatenate(all_true)

    preds = (all_probs >= threshold).astype(int)

    tp = np.sum((preds==1) & (all_true==1))
    fp = np.sum((preds==1) & (all_true==0))
    tn = np.sum((preds==0) & (all_true==0))
    fn = np.sum((preds==0) & (all_true==1))

    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2*precision*recall / (precision + recall + 1e-9)
    acc       = (tp + tn) / (tp + tn + fp + fn + 1e-9)

    # AUC
    auc_metric = BinaryAUROC().to(DEVICE)
    auc = auc_metric(torch.tensor(all_probs).to(DEVICE), torch.tensor(all_true).to(DEVICE)).item()

    return (val_loss / len(loader.dataset)), acc, precision, recall, f1, auc, (tp, fp, tn, fn)

def train_model(model, epochs_head=5, epochs_finetune=12, target_precision=0.97):
    best_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    patience = 5
    bad = 0

    # threshold tuning will be done after training; use 0.5 during training eval
    threshold = 0.5

    def run_epochs(num_epochs, phase_name):
        nonlocal best_wts, best_val_loss, bad

        for epoch in range(num_epochs):
            model.train()
            running_loss = 0.0

            for xb, yb in train_loader:
                xb = xb.to(DEVICE)
                yb = torch.tensor([map_to_pos(int(y)) for y in yb], dtype=torch.float32).to(DEVICE).view(-1, 1)

                optimizer.zero_grad(set_to_none=True)

                with autocast():
                    logits = model(xb)
                    loss = criterion(logits, yb)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                running_loss += loss.item() * xb.size(0)

            train_loss = running_loss / len(train_loader.dataset)
            val_loss, acc, prec, rec, f1, auc, cm = evaluate(model, val_loader, threshold=threshold)

            scheduler.step(val_loss)

            print(f"[{phase_name}] Epoch {epoch+1}/{num_epochs} | "
                  f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                  f"acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} f1={f1:.4f} auc={auc:.4f} "
                  f"cm(tp,fp,tn,fn)={cm}")

            # Save best by val_loss (stable)
            if val_loss < best_val_loss - 1e-4:
                best_val_loss = val_loss
                best_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), "best_resnet18_angiogram.pt")
                bad = 0
            else:
                bad += 1
                if bad >= patience:
                    print("Early stopping triggered.")
                    return

    # Phase 1: train head
    run_epochs(epochs_head, "HEAD")

    # Phase 2: fine-tune last layers for better precision/robustness
    # Unfreeze layer4 + fc
    for name, p in model.named_parameters():
        if "layer4" in name or "fc" in name:
            p.requires_grad = True

    # Lower LR for fine-tuning
    global optimizer, scheduler
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.4, patience=2)

    run_epochs(epochs_finetune, "FINETUNE")

    model.load_state_dict(best_wts)
    return model

model = train_model(model, epochs_head=5, epochs_finetune=12)
print("Training done. Best model saved as best_resnet18_angiogram.pt")



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/tmp/ipython-input-4239979740.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.12/dist-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
